# Chapter 16 &mdash; A BDD Decides SAT by Being Built

**Concept 13 of the Chapter 16 decomposition:** *A BDD Decides SAT by Being Built*

Build the diagram and satisfiability is settled &mdash; an unsatisfiable formula <i>is</i> the single 0 node. No search happens.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-BDD-Decides-SAT/Concept-BDD-Decides-SAT.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Jove carries a **BDD engine** (under `BDD/`, Tyler Sorensen's code). From a notebook:

```python
from jove.Bdd import *
```

A formula is written in a small specification language &mdash; a variable order, named
sub-expressions, and a `Main_Exp` naming the one to build:

```
Var_Order : a b c d
f = a | (b & c & d)
Main_Exp : f
```

Operators are `&`, `|`, `~`, `=>`, `<=>` and `XOR`.

`bdd(spec)` builds the **reduced ordered** diagram and draws it. What matters for this
chapter is what happens *after* it is built, which is **nothing**. The diagram of an
unsatisfiable formula is the single terminal node `0`; satisfiability is settled by
looking at what you have, not by searching it.

That is a different shape of algorithm from the CDCL search of Concept 12, and it
answers a harder question than SAT: it holds **every** satisfying assignment at once,
not one. The catch &mdash; developed over the next five concepts &mdash; is that the
diagram itself can be exponentially large.

## 2. Definitions

### Build one, and look at it

In [ ]:
f = bdd('''
Var_Order : a b c d
f = a | (b & c & d)
Main_Exp : f
''')
f.report()
f

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;12.&nbsp;SAT in Practice: Solvers, DIMACS, and Equisatisfiability](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-SAT-In-Practice/Concept-SAT-In-Practice.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;14.&nbsp;Both Normal Forms, Read Off One Diagram](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Normal-Forms-From-BDD/Concept-Normal-Forms-From-BDD.ipynb)&nbsp;&rarr;

---

## 3. Tests

The object carries its answers as **data**, not as printed text, so a notebook can check them.

In [ ]:
print('variables :', f.vars)
print('nodes     :', f.nodes)
print('satisfying:', f.count, 'of', 2 ** len(f.vars))
print('is_sat    :', f.is_sat)
print('is_taut   :', f.is_taut)
print()
print('one satisfying assignment:', f.models[0])

**An unsatisfiable formula.** All four clauses over two variables, which between them forbid every assignment. Watch what the diagram becomes.

In [ ]:
u = bdd('''
Var_Order : p q
c1 =  p |  q
c2 = ~p |  q
c3 =  p | ~q
c4 = ~p | ~q
Main_Exp : c1 & c2 & c3 & c4
''')
print(repr(u))
print()
print('the whole diagram:')
for line in u.dot.splitlines():
    if line.startswith('Node'):
        print('   ', line)
print()
print('One node, labelled 0.  Nothing was searched -- REDUCING the diagram')
print('and DECIDING satisfiability are the same act.')
assert not u.is_sat and u.count == 0

And a tautology reduces just as hard, to the single `1` node.

In [ ]:
t = bdd('Var_Order : p q\nMain_Exp : (p => q) | (q => p)')
print(repr(t), '  is_taut:', t.is_taut)
assert t.is_taut

## 4. Exercises


1. Write a formula over three variables with exactly one satisfying assignment.
   How many nodes does its diagram have, and why can it not be fewer?
2. `f.models` lists assignments; `paths(f, 1)` (Concept 14) lists *paths*. Build
   `a | b` and explain why the two lists have different lengths.
3. Feed `bdd()` a spec with a syntax error and read the exception. What part of the
   pipeline raised it?
4. The engine takes `Var_Order` as given and does not reorder. Build
   `a | (b & c & d)` under the order `d c b a` and compare node counts. Concept 18
   is about exactly this.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16/Concept-BDD-Decides-SAT')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')